In [1]:
import os
import re
import numpy as np
import pandas as pd
import statistics

import rasterio

from PIL import Image
import matplotlib.pyplot as plt

from pathlib import Path

# DATASETS

Bhargav-1 : Panama, Montijo    - M, D, Y in plants. 2019-09-14 - 2019-12-09   
Bhargav-2 : Panama, Chiriqui_1 - M, D, Y in plants. 2016-01-09 - 2016-11-30  
Sergio    : Belige             - M, D, Y in plots. 2021-09-02 - 2022-03-11  

Jennifer-1: Ghana              - NA. Read me says 2015  
Jennifer-2: El Salvador        - NA. Readme says between 2013 - 2014  

Ben-1     : Costa Rica, Sierpe - Y in plots. 2012  
Ben-2     : Honduras, Blanca   - NA. Read me says 2016  

Moody-1   : Panama-Chirqui_2   - Y. 2014  
Moody-2   : Costa Rica, Nicoya - Y. 2012  

Brazil-AcarauBoca   : Jan 2016  
Brazil-Manginho     : Jan 2016  
Brazil-Manguezal    : Jan 2016  
Brazil-Rego         : Jan 2016  

Brazil-FuroGrande   : Feb 13, 2017  
Brazil-BocaGrande   : Feb 14, 2017  
Brazil-Furo_Do_Chato: Feb 15, 2017  
Brazil-Mangue       : Feb 16, 2017  
Brazil-Caetano      : Feb 18, 2017  
Brazil-Maruipe      : Feb 21, 2017  
Brazil-Barreto      : Feb 21, 2017  
Brazil-Salinas      : Feb 22, 2017  

## PREDETERMINED TIMERANGE.
Satellite data is not reliable in the granularity of a day.  
AGB data is expected to change very slowly over a period of  
time. So taking a clear satellite data on any data in the target year may work.  

In [2]:
# Format: {Start date, End date}
timelines = {
     'CostaRica-Nicoya' : ("2012-01-01", "2012-12-31"),
     'CostaRica-Sierpe' : ("2012-01-01", "2012-12-31"),
     'ElSalvador'       : ("2013-01-01", "2013-12-31"),
     'Panama-Chirqui_2' : ("2014-01-01", "2014-12-31"),
     #'Ghana'            : ("2015-01-01", "2015-12-31"), # Removed it, because we are sticking to Central America.

     'Honduras-Blanca'  : ("2016-01-01", "2016-12-31"),
     'Panama-Chirqui_1' : ("2016-01-01", "2016-12-31"),
     'Brazil-AcarauBoca': ("2016-01-01", "2016-12-31"),
     'Brazil-Manginho'  : ("2016-01-01", "2016-12-31"),
     'Brazil-Manguezal' : ("2016-01-01", "2016-12-31"),
     'Brazil-Rego'      : ("2016-01-01", "2016-12-31"),

     'Brazil-FuroGrande': ("2017-01-01", "2017-12-31"),
     'Brazil-BocaGrande': ("2017-01-01", "2017-12-31"),
     'Brazil-Furo_Do_Chato': ("2017-01-01", "2017-12-31"),
     'Brazil-Mangue'    : ("2017-01-01", "2017-12-31"),
     'Brazil-Caetano'   : ("2017-01-01", "2017-12-31"),
     'Brazil-Maruipe'   : ("2017-01-01", "2017-12-31"),
     'Brazil-Barreto'   : ("2017-01-01", "2017-12-31"),
     'Brazil-Salinas'   : ("2017-01-01", "2017-12-31"),

     #'Panama-Montijo'    : ("2019-01-01", "2019-12-31"),
     'Belige'           : ("2021-01-01", "2021-12-01"),
}

# GENRAL CODE

In [3]:
def impute_heights(df):
    # Compute median from all of the height values in the local geography.
    height_median = df['height'].median()

    # Fill NaN with the median height.
    df['height'].fillna(height_median)
    return df

In [4]:
def read_csv(csv_file):
    try:
        df = pd.read_csv(csv_file, encoding='utf-8-sig')
    except Exception as e:
        df = pd.read_csv(csv_file, encoding='cp1252')
    return df

## Function to standardize column names.

In [5]:
uncommon_names = {'total_height': 'height',
                  'tree_height': 'height',
                  'diameter_dbh': 'diameter',
                  'DBH (cm)': 'diameter',
                  'plant_AGB_kg_species': 'plant_AGB_kg',
                  'height_canopy': 'canopy_height',
                  'species_code': 'species',
                  'Species name (scientific)': 'species',
                  'Status (live/1/2/3)': 'alive_or_dead',
                  'Total AGB (kg)': 'plant_AGB_kg',
                  'wood density (g/cm3)': 'wood_density',
                  'Latitude': 'latitude',
                  'Longitude': 'longitude',
                  'plot_center_latitude': 'latitude',
                  'plot_center_longitude': 'longitude',
                  'Sub-plot area (ha)': 'plot_area_ha',
                  'Site ID': 'site_id',
                  'Plot': 'plot_id'}

def standardize_cols(df):
    #column_map = {k.lower(): v.lower() for k, v in uncommon_names.items()}

    # Rename uncommon columns to standard names
    df = df.rename(columns=uncommon_names)
    return df

## Standardize coordinates.

In [6]:
def dms_to_decimal(coord_str):
    """
    Converts degree-minute string to decimal degrees.
    Handles formats like:
        W047 23.654
        S00 37.952
        N12 34.567
        E103 45.123
    Returns float or NaN if already a number or unparseable.
    """
    # If already a number just return it
    try:
        return float(coord_str)
    except (ValueError, TypeError):
        pass

    if not isinstance(coord_str, str):
        return float('nan')

    coord_str = coord_str.strip()

    # Extract direction, degrees, minutes
    match = re.match(r'([NSEW])\s*(\d+)\s+([\d.]+)', coord_str)
    if not match:
        return float('nan')

    direction = match.group(1)
    degrees   = float(match.group(2))
    minutes   = float(match.group(3))

    decimal = degrees + minutes / 60

    if direction in ('S', 'W'):
        decimal = -decimal

    return decimal

def std_coordinates(df, lat_col='latitude', lon_col='longitude'):
    df[lat_col] = df[lat_col].astype(str).apply(dms_to_decimal)
    df[lon_col] = df[lon_col].astype(str).apply(dms_to_decimal)
    return df

## Combine plants data, latitude, and longitude.

In [7]:
# Build lookup hash table.
def build_coord_lookup(sites_df):
    lookup = {}
    for _, row in sites_df.iterrows():
        site_id = row.get('site_id')
        plot_id = row.get('plot_id')

        if plot_id is not None and not pd.isna(plot_id):
            key = (site_id, plot_id)
        else:
            key = site_id

        lookup[key] = (row['latitude'], row['longitude'])
    return lookup


def get_coords(row, lookup):
    # Try (site_id, plot_id) first
    key = (row.get('site_id'), row.get('plot_id'))
    if key in lookup:
        return lookup[key]

    # Fall back to site_id only
    key = row.get('site_id')
    if key in lookup:
        return lookup[key]

    return (None, None)

def merge_long_lat(plants_df, sites_df):
    coord_lookup = build_coord_lookup(sites_df)

    coords = plants_df.apply(lambda row: get_coords(row, coord_lookup), axis=1)
    plants_df['latitude']  = coords.apply(lambda x: x[0])
    plants_df['longitude'] = coords.apply(lambda x: x[1])

    return plants_df

## Code to calcuate AGB in Kg, if this measurement is missing.

Some datasets only contain AGB measured in megagrams per hectare.
They don't have a plant level AGB information in KG.

In such cases, convert MG per hectare to Kgs per tree.  
**Mechanism:**  
 - plot_area_ha      = π × plot_radius² / 10000  
 - total_AGB_Mg      = biomass_aboveground × plot_area_ha  
 - total_AGB_kg      = total_AGB_Mg × 1000  
 - per_tree_AGB_kg   = total_AGB_kg / plot_density  

This gives you an average per-tree value — every tree in the plot gets the same number.  
That is not the same as an individually measured per-tree AGB from an allometric equation.  
It loses all within-plot tree-level variation and would be a rough approximation at best.  

**My assumptions:**
 - The Belize dataset uses plots on 125 meter transects.
 - Within that small area, trees share the same Microclimate, Soil conditions, Tidal inundation frequency, Local hydrology, etc.

So averaging AGB across trees within a plot and treating that as the plot-level AGB is a reasonable aggregation.  

It is highly possible that there could be some kind of within-plot tree variation, but I would argue that such a variation will be much smaller than between-site or between-region variation.

In [8]:
def compute_plant_AGB_kg(df):
    # Step 1: plot area in hectares
    plot_area_ha    = np.pi * df['plot_radius'] ** 2 / 10000

    # Step 2: detect density units
    density_median = df['plot_density'].median()
    if density_median < 50:
        #print(f"  plot_density median = {density_median:.1f} — treating as stems per PLOT")
        stems_per_ha = df['plot_density'] / plot_area_ha
    else:
        #print(f"  plot_density median = {density_median:.1f} — treating as stems per HECTARE")
        stems_per_ha = df['plot_density']

    # Step 3: per-tree AGB in kg
    gms_per_kg = 1000

    #Equation: Mg/ha × ha = Mg
    #          Mg × 1000  = kg
    #          kg ÷ stems = kg/stem
    df['plant_AGB_kg'] = (df['biomass_aboveground'] * plot_area_ha * 1000 / stems_per_ha)

    #print(f"  Min  : {df['plant_AGB_kg'].min():.2f} kg")
    #print(f"  Max  : {df['plant_AGB_kg'].max():.2f} kg")
    #print(f"  Mean : {df['plant_AGB_kg'].mean():.2f} kg")

    return df

# JUST ANALYZE THE COLUMN NAMES

## Panama files

### Bhargav-1

In [13]:
plants_csv = '../../DATA/AGB_DATA/DISCARDED/Panama-Montijo/Romero_et_al_2025_plants.csv'
sites_csv  = '../../DATA/AGB_DATA/DISCARDED/Panama-Montijo/Romero_et_al_2025_sites.csv'

bhargav_1 = merge_long_lat(read_csv(plants_csv), read_csv(sites_csv))
bhargav_1.columns

Index(['study_id', 'site_id', 'plot_id', 'plant_id', 'year', 'month', 'day',
       'plot_radius', 'species', 'diameter_height', 'diameter', 'total_height',
       'decay_class', 'alive_or_dead', 'latitude', 'longitude'],
      dtype='object')

### Bhargav-2

In [14]:
plants_csv = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/Panama-Chirqui_1/Romero_et_al_2025_plants.csv'
sites_csv  = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/Panama-Chirqui_1//Romero_et_al_2025_sites.csv'

bhargav_2 = merge_long_lat(read_csv(plants_csv), read_csv(sites_csv))
bhargav_2.columns

Index(['study_id', 'site_id', 'plot_id', 'plant_id', 'year', 'month', 'day',
       'species', 'plot_radius', 'alive_or_dead', 'decay_class', 'diameter',
       'diameter_height', 'diameter_flag', 'basal_area', 'basal_density',
       'total_height', 'canopy_height', 'height_method', 'canopy_quality',
       'plant_AGB_kg_general', 'plant_AGC_kg_general', 'plant_AGB_kg_species',
       'plant_AGC_kg_species', 'plant_BGB_kg', 'plant_BGC_kg', 'plant_notes',
       'latitude', 'longitude'],
      dtype='object')

**NOTE:** canopy_height canopy_quality are available only in this dataset

### Ben-1

In [15]:
plants_csv = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/CostaRica-Sierpe/Cifuentes_and_Torres_2024_plants.csv'
sites_csv  = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/CostaRica-Sierpe/Cifuentes_and_Torres_2024_plots.csv'

ben_1 = merge_long_lat(read_csv(plants_csv), read_csv(sites_csv))
ben_1.columns

Index(['study_id', 'site_id', 'transect_id', 'plot_id', 'tree_id',
       'plot_radius', 'alive_or_dead', 'family', 'species', 'diameter',
       'basal_area', 'basal_density', 'height', 'wood_density', 'plant_AGB_kg',
       'plant_AGBK_kg', 'plant_BGB_kg', 'plant_AGB_MgHa', 'plant_AGBK_MgHa',
       'plant_BGB_MgHa', 'plant_mass_total_MgHa', 'plant_mass_totalK_MgHa',
       'plant_aboveground_carbon', 'plant_aboveground_carbon_K',
       'plant_belowground_carbon', 'plant_total_carbon',
       'plant_total_carbon_K', 'decay_reduction_factor',
       'carbon_conversion_factor', 'latitude', 'longitude'],
      dtype='object')

### Ben-2

In [16]:
plants_csv = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/Honduras-Blanca/Flores-Marin_and_Cifuentes-Jara_2025_plants.csv'
sites_csv  = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/Honduras-Blanca/Flores-Marin_and_Cifuentes-Jara_2025_plots.csv'

ben_2 = merge_long_lat(read_csv(plants_csv), read_csv(sites_csv))
ben_2.columns

Index(['study_id', 'site_id', 'transect_id', 'plot_id', 'tree_id', 'species',
       'wood_density', 'alive_or_dead', 'diameter', 'diameter_flag', 'height',
       'basal_density', 'trunk_mass', 'branch_mass', 'leaf_mass',
       'aerial_root_mass', 'plant_AGB_kg_compiled', 'plant_BGB_kg',
       'plant_mass_total_kg', 'plant_AGB_kg', 'latitude', 'longitude'],
      dtype='object')

### Moody-1

In [17]:
plants_csv = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/Panama-Chirqui_2/Cifuentes_et_al_2023_plants.csv'
sites_csv  = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/Panama-Chirqui_2/Cifuentes_et_al_2023_plots.csv'

moody_1 = merge_long_lat(read_csv(plants_csv), read_csv(sites_csv))
moody_1.columns

Index(['study_id', 'site_id', 'plot_id', 'plot_radius', 'species', 'family',
       'diameter', 'diameter_flag', 'alive_or_dead', 'decay_class', 'height',
       'wood_density', 'carbon_conversion_factor', 'basal_area',
       'basal_density', 'plant_aboveground_carbon', 'plant_belowground_carbon',
       'plant_BGB_kg', 'plant_AGB_kg', 'plant_AGB_MgHa', 'plant_BGB_MgHa',
       'plant_mass_total_MgHa', 'plant_total_carbon', 'plant_notes',
       'latitude', 'longitude'],
      dtype='object')

### Moody-2

In [18]:
plants_csv = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/CostaRica-Nicoya/Cifuentes_et_al_2024_plants.csv'
sites_csv  = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/CostaRica-Nicoya/Cifuentes_et_al_2024_plots.csv'

moody_2 = merge_long_lat(read_csv(plants_csv), read_csv(sites_csv))
moody_2.columns

Index(['study_id', 'site_id', 'plot_id', 'family', 'species', 'alive_or_dead',
       'plot_radius', 'decay_class', 'diameter', 'diameter_flag',
       'diameter_crown', 'height', 'height_canopy', 'basal_area',
       'basal_density', 'plant_notes', 'wood_density', 'plant_BGB_kg',
       'plant_AGB_kg', 'carbon_conversion_factor', 'plant_belowground_carbon',
       'plant_aboveground_carbon', 'plant_AGB_MgHa', 'plant_BGB_MgHa',
       'plant_mass_total_MgHa', 'plant_total_carbon', 'latitude', 'longitude'],
      dtype='object')

### Jennifer-1

### Jennifer-2

In [19]:
#../../DATA/CCN-Data-Library/data/primary_studies/Cifuentes_et_al_2024_ElSalvador/original/Cifuentes_et_al_2024_ElSalvador_plants.csv
plants_csv = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/ElSalvador/Cifuentes_et_al_2024_ElSalvador_plants.csv'
sites_csv  = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/ElSalvador/Cifuentes_et_al_2024_ElSalvador_plots.csv'

jen_2 = merge_long_lat(read_csv(plants_csv), read_csv(sites_csv))
jen_2.columns

Index(['study_id', 'site_id', 'plot_id', 'transect', 'plant_id', 'family',
       'genus', 'species', 'common_name', 'alive_or_dead', 'decay_class',
       'diameter', 'diameter_crown', 'diameter_notes', 'height',
       'height_canopy', 'plant_AGB_g', 'plant_AGB_Mg', 'plant_AGB_kg',
       'plant_BGB_Mg', 'plant_BGB_kg', 'plant_aboveground_carbon_Mg',
       'plant_belowground_carbon_Mg', 'plant_aboveground_carbon_MgHa',
       'plant_belowground_carbon_MgHa', 'plant_total_carbon', 'latitude',
       'longitude'],
      dtype='object')

### Sergio

**NOTE:**  
This dataset does not have plant level AGB in Kg, but biomass_aboveground exists.

In [20]:
plants_csv = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/Belige/Morrissette_et_al_2023_biomass.csv'
sites_csv  = '../../DATA/AGB_DATA/Original_Data/Non_Brazil/Belige/Morrissette_et_al_2023_plots.csv'

sergio_1 = merge_long_lat(read_csv(plants_csv), read_csv(sites_csv))
sergio_1.columns

Index(['study_id', 'site_id', 'transect_id', 'plot_id', 'plot_radius',
       'species_code', 'biomass_flag', 'decay_class', 'debris_number',
       'plot_density', 'tree_height', 'canopy_width', 'diameter_dbh',
       'diameter_base', 'diameter_qmd', 'wood_volume', 'wood_volume_flag',
       'biomass_decay_corrected', 'decay_3_wood_volume', 'decay_3_biomass_g',
       'decay_3_biomass_Mg', 'decay_3_biomass_aboveground',
       'biomass_aboveground', 'biomass_aboveground_scaled',
       'biomass_aboveground_carbon', 'biomass_downed_wood',
       'biomass_belowground', 'biomass_belowground_scaled', 'biomass_total',
       'biomass_belowground_carbon', 'biomass_total_carbon', 'latitude',
       'longitude'],
      dtype='object')

## Brazil files

In [22]:
brazil_files = list(Path('../../DATA/AGB_DATA/Original_Data/Brazil').rglob('*.xlsx'))
#brazil_files = list(Path('/Users/kosaraju_b/Desktop/NASA_DATA/Ben').glob('*.xlsx'))
#brazil_files

for file in brazil_files:
    print(file)
    try:
        df = pd.read_excel(file, sheet_name='Data')
        print(df.columns)
        print("\n")
    except Exception as e:
        print("NOT readable")

../../DATA/AGB_DATA/Original_Data/Brazil/Brazil-Mangue/SWAMP Data-Trees-Mangue Sul-Brazil-2017.xlsx
Index(['No ID', 'Site ID', 'Data collection date (dd/mm/yyyy)', 'Plot',
       'Sub-plot', 'Latitude', 'Longitude', 'Species name (scientific)',
       'Species name (local)', 'DBH (cm)', 'Status (live/1/2/3)', 'notes',
       'wood density (g/cm3)', 'volume (cm3)', 'Wood Mass AG (kg)',
       'Total AGB (kg)', 'BG root mass (kg)', 'Sub-plot area (ha)',
       'Sub-plot design', 'AGB (Mg/ha)', 'BGB (Mg/ha)', 'AGC (MgC/ha)',
       'BGC (MgC/ha)', 'Basal area per ha ', 'AGB summed per plot (Mg/ha)',
       'BGB summed per plot (Mg/ha)', 'AGC summed per plot (MgC/ha)',
       'BGC summed per plot (MgC/ha)', 'Basal area (m2/ha) summed per plot',
       'source for density', 'source for allometry'],
      dtype='object')


../../DATA/AGB_DATA/Original_Data/Brazil/Brazil-Furo_Do_Chato/SWAMP Data-Trees-Furo do Chato-Brazil-2017.xlsx
Index(['No ID', 'Site ID', 'Data collection date (dd/mm/yyyy)

**NOTE:** These datasets does not have **height** column.

# PROCESS THE DATA

In [23]:
BASE_PATH = "../../DATA/AGB_DATA/Original_Data/"

brazil_path     = Path(BASE_PATH + "Brazil")
non_brazil_path = Path(BASE_PATH + "Non_Brazil")

## PROCESS NON-BRAZIL FILES

In [24]:
files = []

for directory in non_brazil_path.iterdir():
    #print(f"{directory.name}")

    plants_files = list(directory.glob('*plants*.csv')) or list(directory.glob('*biomass*.csv'))
    plots_files = list(directory.glob('*plots*.csv')) or list(directory.glob('*sites*.csv'))

    pairs = [(plants, plots) for plants, plots in zip(plants_files, plots_files) if plants and plots]
    files += pairs

In [25]:
for plants, plots in files:
    print(f"plants: {plants}")
    print(f"plots : {plots}")

plants: ../../DATA/AGB_DATA/Original_Data/Non_Brazil/ElSalvador/Cifuentes_et_al_2024_ElSalvador_plants.csv
plots : ../../DATA/AGB_DATA/Original_Data/Non_Brazil/ElSalvador/Cifuentes_et_al_2024_ElSalvador_plots.csv
plants: ../../DATA/AGB_DATA/Original_Data/Non_Brazil/Panama-Chirqui_2/Cifuentes_et_al_2023_plants.csv
plots : ../../DATA/AGB_DATA/Original_Data/Non_Brazil/Panama-Chirqui_2/Cifuentes_et_al_2023_plots.csv
plants: ../../DATA/AGB_DATA/Original_Data/Non_Brazil/CostaRica-Nicoya/Cifuentes_et_al_2024_plants.csv
plots : ../../DATA/AGB_DATA/Original_Data/Non_Brazil/CostaRica-Nicoya/Cifuentes_et_al_2024_plots.csv
plants: ../../DATA/AGB_DATA/Original_Data/Non_Brazil/CostaRica-Sierpe/Cifuentes_and_Torres_2024_plants.csv
plots : ../../DATA/AGB_DATA/Original_Data/Non_Brazil/CostaRica-Sierpe/Cifuentes_and_Torres_2024_plots.csv
plants: ../../DATA/AGB_DATA/Original_Data/Non_Brazil/Belige/Morrissette_et_al_2023_biomass.csv
plots : ../../DATA/AGB_DATA/Original_Data/Non_Brazil/Belige/Morrissette_e

In [26]:
panama_count = 0
all_datasets = []

for (plant_csv, site_csv) in files:
    folder_name = Path(plant_csv).parent.name
    print("Processing: %s ..." %folder_name)
    start_date, end_date = timelines[folder_name]

    plants_df = read_csv(plant_csv)
    plants_df = standardize_cols(plants_df)

    # REVISIT: Shall we impute NA values in "height" column?
    # plants_df = impute_heights(plants_df)

    sites_df = read_csv(site_csv)
    sites_df = standardize_cols(sites_df)

    df = merge_long_lat(plants_df, sites_df)
    df.attrs['name'] = folder_name
    df['dataset'] = folder_name
    df['start_date'] = start_date
    df['end_date']   = end_date

    panama_count += len(df)
    all_datasets.append(df)

print("Panama count: %d" %panama_count)

Processing: ElSalvador ...
Processing: Panama-Chirqui_2 ...
Processing: CostaRica-Nicoya ...
Processing: CostaRica-Sierpe ...
Processing: Belige ...
Processing: Panama-Chirqui_1 ...
Processing: Honduras-Blanca ...
Panama count: 18141


## PROCESS BRAZIL FILES

**NOTE:** Brazil datasets does not have **height** column.
I do not want to throw away the entire dataset just because **height** parameter is not there.

**Though process:**
 - Create a height column with NaN.
 - Merge the Brazil data with Panama data.
 - Compute the median height from among the total merged rows.

In [27]:
brazil_files = []
for directory in brazil_path.iterdir():
    print(f"{directory.name}")
    found_files = list(directory.glob('*.xlsx'))
    brazil_files += [file for file in found_files if file]
brazil_files

.DS_Store
Brazil-Mangue
Brazil-Furo_Do_Chato
Brazil-Manguezal
Brazil-Maruipe
Brazil-AcarauBoca
Brazil-BocaGrande
Brazil-Barreto
Brazil-Manginho
Brazil-Salinas
Brazil-FuroGrande
Brazil-Rego
Brazil-Caetano


[PosixPath('../../DATA/AGB_DATA/Original_Data/Brazil/Brazil-Mangue/SWAMP Data-Trees-Mangue Sul-Brazil-2017.xlsx'),
 PosixPath('../../DATA/AGB_DATA/Original_Data/Brazil/Brazil-Furo_Do_Chato/SWAMP Data-Trees-Furo do Chato-Brazil-2017.xlsx'),
 PosixPath('../../DATA/AGB_DATA/Original_Data/Brazil/Brazil-Manguezal/SWAMP Data-Trees-Manguezal Cauassu-Brazil-2016.xlsx'),
 PosixPath('../../DATA/AGB_DATA/Original_Data/Brazil/Brazil-Maruipe/SWAMP Data-Trees-Maruipe-Brazil-2017.xlsx'),
 PosixPath('../../DATA/AGB_DATA/Original_Data/Brazil/Brazil-AcarauBoca/SWAMP Data-Trees-Acarau Boca-Brazil-2016.xlsx'),
 PosixPath('../../DATA/AGB_DATA/Original_Data/Brazil/Brazil-BocaGrande/SWAMP Data-Trees-Boca Grande-Brazil-2017.xlsx'),
 PosixPath('../../DATA/AGB_DATA/Original_Data/Brazil/Brazil-Barreto/SWAMP Data-Trees-Barreto-Brazil-2017.xlsx'),
 PosixPath('../../DATA/AGB_DATA/Original_Data/Brazil/Brazil-Manginho/SWAMP Data-Trees-Manginho-Brazil-2016.xlsx'),
 PosixPath('../../DATA/AGB_DATA/Original_Data/Brazil/B

In [28]:
assert all_datasets

iterate = 0
brazil_row_count = 0

for file in brazil_files:
    folder_name = Path(file).parent.name
    print("Processing: %s ..." %folder_name)
    start_date, end_date = timelines[folder_name]

    try:
        brazil_df = pd.read_excel(file, sheet_name='Data')

        #Standardize the column names.
        brazil_df = standardize_cols(brazil_df)

        # Calculate height from medians.
        # NOTE: It appears that Height was never collected in the Brazilian protocol.
        brazil_df['height'] = None
        brazil_df['dataset'] = folder_name
        brazil_df['start_date'] = start_date
        brazil_df['end_date']   = end_date

        brazil_df.attrs['name'] = folder_name
        brazil_row_count += len(brazil_df)

        all_datasets.append(brazil_df)
        iterate += 1
    except Exception as e:
        print("%s: NOT readable" %file)

print("Brazil row count: " + str(brazil_row_count))

Processing: Brazil-Mangue ...
Processing: Brazil-Furo_Do_Chato ...
Processing: Brazil-Manguezal ...
Processing: Brazil-Maruipe ...
Processing: Brazil-AcarauBoca ...
Processing: Brazil-BocaGrande ...
Processing: Brazil-Barreto ...
Processing: Brazil-Manginho ...
Processing: Brazil-Salinas ...
Processing: Brazil-FuroGrande ...
Processing: Brazil-Rego ...
Processing: Brazil-Caetano ...
Brazil row count: 957


In [29]:
# DEBUG ONLY
global_count = 0
for idx in range(0, len(all_datasets)):
    dataset = all_datasets[idx]
    global_count += len(dataset)
    print("idx: %d, Name: %s, count: %d" %(idx, dataset.attrs['name'], len(dataset)))

print("Total count: %d" %global_count)
print("Panama count: %d" %panama_count)

idx: 0, Name: ElSalvador, count: 6300
idx: 1, Name: Panama-Chirqui_2, count: 578
idx: 2, Name: CostaRica-Nicoya, count: 1886
idx: 3, Name: CostaRica-Sierpe, count: 3245
idx: 4, Name: Belige, count: 3934
idx: 5, Name: Panama-Chirqui_1, count: 1401
idx: 6, Name: Honduras-Blanca, count: 797
idx: 7, Name: Brazil-Mangue, count: 91
idx: 8, Name: Brazil-Furo_Do_Chato, count: 53
idx: 9, Name: Brazil-Manguezal, count: 84
idx: 10, Name: Brazil-Maruipe, count: 65
idx: 11, Name: Brazil-AcarauBoca, count: 60
idx: 12, Name: Brazil-BocaGrande, count: 71
idx: 13, Name: Brazil-Barreto, count: 60
idx: 14, Name: Brazil-Manginho, count: 134
idx: 15, Name: Brazil-Salinas, count: 84
idx: 16, Name: Brazil-FuroGrande, count: 52
idx: 17, Name: Brazil-Rego, count: 112
idx: 18, Name: Brazil-Caetano, count: 91
Total count: 19098
Panama count: 18141


# PREPROCESS THE DATASETS

## Standardize coordinates.

In [30]:
for idx in range(0, len(all_datasets)):
    all_datasets[idx] = std_coordinates(all_datasets[idx],
                                        lat_col='latitude',
                                        lon_col='longitude')

## Calcuate Plant_AGB_Kg if not present

In [31]:
for idx in range(0, len(all_datasets)):
    df = all_datasets[idx]
    if "plant_AGB_kg" not in df.columns and \
        "biomass_aboveground" in df.columns:
        df = compute_plant_AGB_kg(df)
    all_datasets[idx] = df

## Compute height for missing rows for Brazil data.
**NOTE:** Check "PROCESS BRAZIL FILES" section for explanation

In [32]:
# Brazil protocol appears to skip "height" as a parameter.
# Compute median from the entire dataset.

# ANALOGY:
#  - Missing height values were imputed using the median height computed from all trees in the combined
#    dataset for which height was measured.
#  - All datasets in this study were collected within a geographically constrained region bounded by
#    [-2.8503]°N to [18.3423]°N latitude and  [-88.9556]°W to [-40.0312]°W longitude.
#  — It is assumed that these regios sharing similar macroclimate, tidal changes, species composition,
#    making regional pooling a reasonable basis for imputation.

#  - We acknowledge this as an approximation.
#    However, the majority of allometric equations applied across datasets in this collection estimate
#    AGB primarily from stem diameter and wood density, with height playing a secondary role.

#  - The influence of imputed height values on final AGB estimates is therefore limited, and this
#    approximation is unlikely to introduce systematic bias into model predictions.

# NOTE: Revisit this analogy.

all_heights = pd.concat([df['height'] for df in all_datasets])

# Compute median from pooled values
height_median = all_heights.median()

count_again = 0
# Fill NaN with the median height.
#for idx in range(0, len(all_datasets)):
for idx in range(7, len(all_datasets)):
    count_again += len(all_datasets[idx])
    all_datasets[idx]['height'] = all_datasets[idx]['height'].fillna(height_median)

print("Entire dataset including Brazil: "+str(count_again))

Entire dataset including Brazil: 957


/var/folders/b8/8qlj5yzd0jng0b9470zpnl9h0000gn/T/ipykernel_42180/1351677806.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_datasets[idx]['height'] = all_datasets[idx]['height'].fillna(height_median)


# SELECT DATASETS
Minimum viable columns needed to predict AGB are:
 - **diameter** : dominant predictor, non-negotiable
 - **height**  : second strongest predictor, non-negotiable
 - **species** : critical for species-specific allometric relationships
 - **alive_or_dead**: needed to separate living from dead trees
 - **latitude** : needed for EO extraction
 - **longitude** : needed for EO extraction

## Define minimum required columns

In [33]:
required_cols = [
    'site_id',
    'plot_id',
    'diameter',
    'height',
    'species',
    'plant_AGB_kg',
    'latitude',
    'longitude',
    'start_date',
    'end_date'
    #'alive_or_dead', # Perhaps use this for the dead/decay type of AGB
]

required_col_set   = set(required_cols)
required_col_count = len(required_cols)

## Select the useful datasets and merge them.

In [34]:
selected_data_df = pd.DataFrame(columns=required_cols)

new_data_set = []
for idx in range(0, len(all_datasets)):
    matched   = required_col_set & set(all_datasets[idx].columns)
    unmatched = required_col_set - set(all_datasets[idx].columns)

    if required_col_count == len(matched):
        #print("Dataset: <" + all_datasets[idx].attrs['name'] + "> is useful")

        # Copy out only the required columns.
        df = all_datasets[idx].filter(items=required_col_set)
        df['dataset'] = all_datasets[idx]["dataset"]
        selected_data_df = pd.concat([selected_data_df, df])

        new_data_set.append(df)
    else:
        print("Dataset: <" + all_datasets[idx].attrs['name'] + "> is NOT useful. Missing: " + str(unmatched))

/var/folders/b8/8qlj5yzd0jng0b9470zpnl9h0000gn/T/ipykernel_42180/3594290558.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  selected_data_df = pd.concat([selected_data_df, df])


In [35]:
print(f"\nTotal rows in selected data: {len(selected_data_df):,}")


Total rows in selected data: 19,098


# Clean up data

### Create a concatenation of site_id and plot_id columns.

In [36]:
import re

def make_unique_plot_id(row):
    site = str(row['site_id'])
    plot = str(row['plot_id'])

    # Check if plot_id matches pattern: site_id followed by underscore and anything
    pattern = re.compile(rf'^{re.escape(site)}_\S+')

    if pattern.match(plot):
        return plot
    else:
        return f"{site}_{plot}"

selected_data_df['plot_id'] = selected_data_df.apply(make_unique_plot_id, axis=1)

# Check that there are no more numerics along in the plot_id column
assert df['plot_id'].str.match(r'^\D+$').all()

### Reorder columns for better readability.

In [37]:
cols_to_move = ['dataset', 'plot_id', 'start_date', 'end_date', 'latitude', 'longitude']
new_order = cols_to_move + [c for c in selected_data_df.columns if c not in cols_to_move]
selected_data_df = selected_data_df[new_order]

### CHECK FOR NAs

In [38]:
def check_na(col, dataset_list):
    for idx in range(0, len(dataset_list)):
        dataset = dataset_list[idx]
        if (dataset[col].isna().sum()):
            print("idx: %d, feature: %s, name: %s, na count: %d" %(idx,
                                                                   col,
                                                                   dataset.attrs['name'],
                                                                   dataset[col].isna().sum()))

    print("")

In [39]:
check_na("height", new_data_set)
check_na("latitude", new_data_set)
check_na("longitude", new_data_set)
check_na("diameter", new_data_set)
check_na("species", new_data_set)
check_na("plant_AGB_kg", new_data_set)

idx: 1, feature: height, name: Panama-Chirqui_2, na count: 495
idx: 2, feature: height, name: CostaRica-Nicoya, na count: 1504
idx: 3, feature: height, name: CostaRica-Sierpe, na count: 3152
idx: 4, feature: height, name: Belige, na count: 52

idx: 0, feature: latitude, name: ElSalvador, na count: 166
idx: 14, feature: latitude, name: Brazil-Manginho, na count: 134
idx: 17, feature: latitude, name: Brazil-Rego, na count: 112

idx: 0, feature: longitude, name: ElSalvador, na count: 166
idx: 14, feature: longitude, name: Brazil-Manginho, na count: 134
idx: 17, feature: longitude, name: Brazil-Rego, na count: 112

idx: 0, feature: diameter, name: ElSalvador, na count: 727
idx: 4, feature: diameter, name: Belige, na count: 48

idx: 4, feature: species, name: Belige, na count: 50

idx: 0, feature: plant_AGB_kg, name: ElSalvador, na count: 3277
idx: 2, feature: plant_AGB_kg, name: CostaRica-Nicoya, na count: 69
idx: 4, feature: plant_AGB_kg, name: Belige, na count: 50
idx: 5, feature: plant_

### Carve out rows that have AGB but are missing height or diameter.

These rows have known AGB values but are missing structural measurements.  
Using them as a held-out validation set gives you an independent check on model  
accuracy that is completely separate from your train/test split.  
Since these rows were excluded from training for data quality reasons, they represent  
a realistic stress test of how well the model generalizes to imperfect real-world data.

In [40]:
validation_df = selected_data_df[(selected_data_df['plant_AGB_kg'].notna()) &
                                (selected_data_df['height'].isna() | selected_data_df['diameter'].isna())
                                ].copy()

# We don't need "site_id" anymore.
validation_df = validation_df.drop(columns=['site_id'])

In [41]:
validation_df.head()

,dataset,plot_id,start_date,end_date,latitude,longitude,diameter,height,species,plant_AGB_kg
0,Panama-Chirqui_2,Batipa_1,2014-01-01,2014-12-31,8.30711,-82.2813,6.0,NaN,Rhizophora mangle,16.05155
1,Panama-Chirqui_2,Batipa_1,2014-01-01,2014-12-31,8.30711,-82.2813,7.0,NaN,Rhizophora mangle,20.96051
2,Panama-Chirqui_2,Batipa_1,2014-01-01,2014-12-31,8.30711,-82.2813,7.0,NaN,Rhizophora mangle,20.96051
3,Panama-Chirqui_2,Batipa_1,2014-01-01,2014-12-31,8.30711,-82.2813,9.4,NaN,Rhizophora mangle,34.91576
4,Panama-Chirqui_2,Batipa_1,2014-01-01,2014-12-31,8.30711,-82.2813,11.0,NaN,Rhizophora mangle,45.83402


In [42]:
len(validation_df)

5155

#### Write the validation data into a separate CSV.

In [44]:
VALIDATE_FILE = "../../DATA/AGB_DATA/Merged_Data/AGB_VALIDATE.csv"
validation_df.to_csv(VALIDATE_FILE, index=False)
print(f"\nSaved to: {VALIDATE_FILE}")


Saved to: ../../DATA/AGB_DATA/Merged_Data/AGB_VALIDATE.csv


### Remove rows with NA in the required columns.

In [45]:
merged_df = selected_data_df.dropna(subset=required_cols).copy()
print(f"\nTotal rows after cleanup: {len(merged_df):,}")


Total rows after cleanup: 8,774


### List all of the Geographies included in the dataset.

In [46]:
list(merged_df["dataset"].unique())

['ElSalvador',
 'Panama-Chirqui_2',
 'CostaRica-Nicoya',
 'CostaRica-Sierpe',
 'Belige',
 'Panama-Chirqui_1',
 'Honduras-Blanca',
 'Brazil-Mangue',
 'Brazil-Furo_Do_Chato',
 'Brazil-Manguezal',
 'Brazil-Maruipe',
 'Brazil-AcarauBoca',
 'Brazil-BocaGrande',
 'Brazil-Barreto',
 'Brazil-Salinas',
 'Brazil-FuroGrande',
 'Brazil-Caetano']

### Bounding box of the geographical area.

In [47]:
print(f"Latitude  : {merged_df['latitude'].min():.4f} to {merged_df['latitude'].max():.4f}")
print(f"Longitude : {merged_df['longitude'].min():.4f} to {merged_df['longitude'].max():.4f}")

Latitude  : -2.8503 to 18.3423
Longitude : -88.9556 to -40.0312


### Show a sample of the dataset.

In [48]:
merged_df.head()

,dataset,plot_id,start_date,end_date,latitude,longitude,site_id,diameter,height,species,plant_AGB_kg
0,ElSalvador,El_Jobal_9_6,2013-01-01,2013-12-31,13.22736,-88.60437,El_Jobal,26.3,12.0,mangle,630.627596
1,ElSalvador,El_Jobal_9_6,2013-01-01,2013-12-31,13.22736,-88.60437,El_Jobal,28.6,14.0,mangle,784.223246
2,ElSalvador,El_Jobal_9_6,2013-01-01,2013-12-31,13.22736,-88.60437,El_Jobal,19.2,4.0,mangle,139.136548
3,ElSalvador,El_Jobal_9_6,2013-01-01,2013-12-31,13.22736,-88.60437,El_Jobal,6.2,8.0,mangle,14.726733
4,ElSalvador,El_Jobal_9_6,2013-01-01,2013-12-31,13.22736,-88.60437,El_Jobal,5.2,5.0,mangle,9.321734


## Write the merged data to the target file.

In [49]:
# We don't need "site_id" anymore.
merged_df = merged_df.drop(columns=['site_id'])

In [51]:
OUTPUT_FILE = "../../DATA/AGB_DATA/Merged_Data/AGB_MERGED.csv"
merged_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved to: {OUTPUT_FILE}")


Saved to: ../../DATA/AGB_DATA/Merged_Data/AGB_MERGED.csv
